# 06 — Target Feasibility: y_addition_180d

**Objetivo:** medir si el primer target puede construirse sin confundir censura, ambigüedad temporal ni eventos previos a t0.

Este notebook es un **laboratorio reproducible**. La lógica reutilizable vive en `src/argos/`;
aquí solo se parametriza, ejecuta, inspecciona evidencia y documentan decisiones.

## Target candidato
`y_addition_180d = 1` si existe una adición válida estrictamente posterior a `t0` y dentro de 180 días.

Un contrato solo puede ser negativo si el horizonte completo está observado.
En esta fase el objetivo es **GO / REDEFINE**, no entrenar un modelo.

In [1]:
from pathlib import Path
import tomllib
import pandas as pd

from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
DATASETS, RELATIONSHIPS = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)

with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

{k: p.name for k, p in FILES.items()}

{'processes': 'SECOP_II_-_Procesos_de_Contratación_20260921.csv',
 'contracts': 'SECOP_II_-_Contratos_Electrónicos_20260921.csv',
 'bidders': 'Proponentes_por_Proceso_SECOP_II_20260921.csv',
 'suppliers': 'SECOP_II_-_Proveedores_Registrados_20260921.csv',
 'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'sanctions': 'SECOPII_-_Multas_y_Sanciones_20260921.csv',
 'payments': 'SECOP_II_-_Plan_de_pagos_20260921.csv'}

In [2]:
from argos.common.reports import write_report
from argos.targets.addition import analyze_addition_target_feasibility, addition_target_by_year

cfg = PROJECT_CONFIG["profiling"]
temporal_cfg = PROJECT_CONFIG["temporal"]
target_cfg = PROJECT_CONFIG["target"]
included_types = tuple(target_cfg.get("included_addition_types", [])) or None

horizon_summary, target_diagnostics = analyze_addition_target_feasibility(
    FILES["contracts"],
    FILES["additions"],
    contract_id_column=target_cfg["contract_id_column"],
    t0_column=target_cfg["candidate_t0_column"],
    addition_contract_id_column=target_cfg["addition_contract_id_column"],
    addition_event_column=target_cfg["addition_event_column"],
    addition_type_column=target_cfg.get("addition_type_column"),
    included_addition_types=included_types,
    horizons=(30, 90, 180, 365),
    date_order=temporal_cfg["date_order"],
    plausible_t0_year_min=2015,
    plausible_t0_year_max=temporal_cfg["plausible_year_max"],
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    horizon_summary,
    paths.reports / "profiling" / "target_feasibility_horizons.parquet",
)
write_report(
    target_diagnostics,
    paths.reports / "profiling" / "target_feasibility_diagnostics.parquet",
)

horizon_summary


,horizon_days,population,eligible,censored,positives,negatives,prevalence,label_cutoff
0,30,5640278,5512909,127369,349240,5163669,0.063349,2026-09-09
1,90,5640278,5361355,278923,869255,4492100,0.162133,2026-09-09
2,180,5640278,5312182,328096,1630300,3681882,0.306898,2026-09-09
3,365,5640278,4514404,1125874,2102645,2411759,0.465764,2026-09-09


In [3]:
# Critical diagnostics now separate true missing t0 from parser failures and
# quantify source-event duplication before target construction.
target_diagnostics.T


,0
contract_source_rows,6066730
distinct_contract_ids,6066730
contract_ids_raw_missing_t0,426452
contract_ids_unparsable_t0,0
contract_ids_without_parsed_t0,426452
contract_ids_ambiguous_t0,0
unambiguous_contracts_with_t0,5640278
plausible_contracts_with_t0,5640278
contracts_t0_after_label_cutoff,26726
addition_source_rows,28378604


In [4]:
by_year = addition_target_by_year(
    FILES["contracts"],
    FILES["additions"],
    horizon_days=target_cfg["horizon_days"],
    contract_id_column=target_cfg["contract_id_column"],
    t0_column=target_cfg["candidate_t0_column"],
    addition_contract_id_column=target_cfg["addition_contract_id_column"],
    addition_event_column=target_cfg["addition_event_column"],
    addition_type_column=target_cfg.get("addition_type_column"),
    included_addition_types=included_types,
    date_order=temporal_cfg["date_order"],
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)

write_report(
    by_year,
    paths.reports / "profiling" / "target_feasibility_by_year.parquet",
)

by_year


,t0_year,eligible,positives,prevalence
0,2015,10,2,0.200000
1,2016,1342,231,0.172131
2,2017,22259,4656,0.209174
3,2018,142973,41560,0.290684
4,2019,142592,47716,0.334633
5,2020,357251,100520,0.281371
6,2021,561581,153749,0.273779
7,2022,710534,229965,0.323651
8,2023,843059,260191,0.308627
9,2024,950670,292046,0.307200


## Decisión obligatoria al finalizar
Documenta en `docs/target-specs/y_addition_180d.md`:
1. `t0` aprobado o rechazado.
2. timestamp del evento.
3. cutoff de etiquetas.
4. población excluida y por qué.
5. prevalencia global y por año.
6. cantidad de observaciones censuradas.
7. definición exacta de qué tipos de adición cuentan.
8. decisión **GO / REDEFINE TARGET**.

Si estas condiciones no están resueltas, no pases a Silver/feature engineering por inercia.

In [5]:
from argos.targets.addition import addition_event_type_profile

event_type_profile = addition_event_type_profile(
    FILES["additions"],
    addition_contract_id_column=target_cfg["addition_contract_id_column"],
    addition_event_column=target_cfg["addition_event_column"],
    addition_type_column=target_cfg.get("addition_type_column", "tipo"),
    date_order=temporal_cfg["date_order"],
    sample_size=cfg["csv_sample_size"],
    temp_directory=paths.root / ".cache" / "duckdb",
)
write_report(
    event_type_profile,
    paths.reports / "profiling" / "addition_event_type_profile.parquet",
)
event_type_profile


,event_type,source_rows,distinct_contracts,parsed_event_rows,min_event_at,max_event_at
0,MODIFICACION GENERAL,13490619,2024452,13490619,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
1,CONCLUSION,7022896,1914612,7022896,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
2,<NULL>,6024441,2090124,6024441,2015-09-02 00:00:00-05:00,2026-09-09 00:00:00-05:00
3,ADICION EN EL VALOR,561997,198072,561997,2015-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
4,CESION,477761,87243,477761,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
5,REACTIVACIoN,429798,89849,429798,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
6,SUSPENSIoN,370555,81192,370555,2016-01-01 00:00:00-05:00,2026-09-09 00:00:00-05:00
7,EXTENSION,391,341,391,2016-01-01 00:00:00-05:00,2018-01-03 00:00:00-05:00
8,REDUCCION EN EL VALOR,145,130,145,2016-01-01 00:00:00-05:00,2017-09-08 00:00:00-05:00
9,EXPIRACION,1,1,1,2017-02-01 00:00:00-05:00,2017-02-01 00:00:00-05:00


## Target interpretation

The feasibility target uses **distinct `(contract_id, event_at, event_type)` events**. Repeated Bronze rows therefore cannot inflate the binary label. The event scope remains all rows in the dedicated Adiciones source until the semantics of `Tipo` justify a narrower target specification.
